# Step 1: The Local Preprocessing Pipeline.

In [ ]:
import os
import json
import math
import time
import concurrent.futures
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import GroupShuffleSplit
from tqdm.notebook import tqdm

In [17]:
def build_feature_pipeline(parquet_path, selected_indices, nose_index=17):
    """
    Reads raw ASL parquet, filters landmarks, normalizes coordinates to the nose, 
    and appends Lag-1 and Lag-2 temporal features.
    
    Args:
        parquet_path (str): Path to the raw parquet file.
        selected_indices (np.array): Array of 130 integer indices to extract.
        nose_index (int): The index of the nose WITHIN the selected_indices array.
        
    Returns:
        np.ndarray: Processed tensor of shape (Frames, 130, 9)
    """
    # 1. Load and Reshape
    df = pd.read_parquet(parquet_path)
    
    # The raw dataset guarantees exactly 543 landmarks per frame in order.
    # We reshape directly to (Frames, 543, 3) for the (x, y, z) columns.
    coords = df[['x', 'y', 'z']].values.reshape(-1, 543, 3)
    
    # 2. Filter to Hoyeol's 130 essential landmarks
    filtered_coords = coords[:, selected_indices, :]  # Shape: (T, 130, 3)
    
    # 3. Handle Missing Values (MediaPipe returns NaN when hands/pose exit the frame)
    # We track where the data actually exists so we don't accidentally normalize ghost coordinates.
    valid_mask = ~np.isnan(filtered_coords[..., 0:1])  # Shape: (T, 130, 1)
    filtered_coords = np.nan_to_num(filtered_coords, nan=0.0)
    
    # 4. Translation Invariance (Normalization)
    # Extract the nose coordinates for every frame. Shape: (T, 1, 3)
    nose_coords = filtered_coords[:, nose_index, :].reshape(-1, 1, 3)
    
    # Subtract nose from all landmarks, but ONLY where the landmark originally existed.
    # Otherwise, missing hands (0,0,0) minus the nose creates a fake vector pointing to the origin.
    normalized_coords = np.where(valid_mask, filtered_coords - nose_coords, 0.0)
    
    # 5. Motion Lags (Velocity and Acceleration equivalents)
    # Lag 1: Frame[t] - Frame[t-1]
    lag1 = np.zeros_like(normalized_coords)
    lag1[1:] = normalized_coords[1:] - normalized_coords[:-1]
    
    # Lag 2: Frame[t] - Frame[t-2]
    lag2 = np.zeros_like(normalized_coords)
    lag2[2:] = normalized_coords[2:] - normalized_coords[:-2]
    
    # Clean up any potential artifacts at the boundaries where differences involve zeroed frames
    lag1 = np.where(valid_mask, lag1, 0.0)
    lag2 = np.where(valid_mask, lag2, 0.0)
    
    # 6. Channel Concatenation
    # Stack [X, Y, Z, dX1, dY1, dZ1, dX2, dY2, dZ2] 
    # Final Shape: (T, 130, 9)
    final_features = np.concatenate([normalized_coords, lag1, lag2], axis=-1)
    
    return final_features.astype(np.float32)

# --- EXECUTION ---
# For your local testing, create a dummy selected_indices array of length 130.
# Ensure you map the standard MediaPipe Nose (index 0 in the raw 543) to whatever 
# position it ends up in your 130-length array (e.g., index 17).

In [18]:
import pandas as pd
import numpy as np

# 1. Load the training manifest from the parent directory
train_df = pd.read_csv('../data/raw/train.csv')

# 2. Randomly sample one row
random_row = train_df.sample(n=1).iloc[0]
raw_path = random_row['path']

# 3. Construct the relative path from the notebooks directory
parquet_path = f"../data/raw/{raw_path}"

print(f"Testing with file: {parquet_path}")

# 4. Create a dummy array of 130 indices for local testing
# We manually assign the raw nose index (0) to position 17 in our subset
selected_indices = np.arange(130)
selected_indices[17] = 0 
nose_idx = 17

# 5. Run the pipeline
final_features = build_feature_pipeline(parquet_path, selected_indices, nose_index=nose_idx)

print(f"Final output shape: {final_features.shape}")

Testing with file: ../train_landmark_files/28656/1996864518.parquet
Final output shape: (23, 130, 9)


In [ ]:
# 1. Setup Directories and Load Manifest
TRAIN_CSV_PATH = '../data/raw/train.csv'
SAVE_DIR = '../data/processed/processed_features'

os.makedirs(SAVE_DIR, exist_ok=True)
train_df = pd.read_csv(TRAIN_CSV_PATH)

selected_indices = np.arange(130) 
selected_indices[17] = 0
NOSE_IDX = 17

# 2. Worker Function
def process_and_save(args):
    raw_path_suffix, sequence_id = args
    raw_path = f"../data/raw/{raw_path_suffix}"
    save_path = os.path.join(SAVE_DIR, f"{sequence_id}.npy")
    
    # Skip if already processed
    if os.path.exists(save_path):
        return True
        
    try:
        features = build_feature_pipeline(raw_path, selected_indices, nose_index=NOSE_IDX)
        np.save(save_path, features.astype(np.float32))
        return True
    except Exception as e:
        print(f"Error processing {raw_path}: {e}")
        return False

print(f"Starting threaded batch preprocessing of {len(train_df)} files...")

tasks = list(zip(train_df['path'], train_df['sequence_id']))

# 3. Execute with Multithreading
# Limiting to 8 workers prevents disk I/O bottlenecking on your NVMe drive
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    results = list(tqdm(executor.map(process_and_save, tasks), total=len(tasks)))

print(f"Successfully processed {sum(results)} / {len(tasks)} files.")
print(f"All features saved to: {SAVE_DIR}")

# Step 2: The ECA and DepthwiseConv1D Blocks

In [19]:
# --- FAILSAFE 1: ENABLE GPU MEMORY GROWTH ---
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Success: Memory Growth Enabled on {len(physical_devices)} GPU(s).")
    except RuntimeError as e:
        print(f"Memory Growth Error: {e}")

# --- MODULES ---
class ECA(layers.Layer):
    """
    Efficient Channel Attention (ECA) Module.
    """
    def __init__(self, kernel_size=5, **kwargs):
        super().__init__(**kwargs)
        self.conv = layers.Conv1D(1, kernel_size=kernel_size, padding='same', use_bias=False)

    def call(self, inputs):
        x = tf.reduce_mean(inputs, axis=1, keepdims=True)
        x = tf.transpose(x, perm=[0, 2, 1])
        x = self.conv(x)
        x = tf.nn.sigmoid(x)
        x = tf.transpose(x, perm=[0, 2, 1])
        return inputs * x

class Conv1DBlock(layers.Layer):
    """
    Depthwise-separable 1D Convolution Block with ECA and Residual Connection.
    Updated for local GPU stability.
    """
    def __init__(self, dim, ksize=17, drop_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        
        # --- FAILSAFE 2: EXPLICIT CAUSAL PADDING + DEPTHWISE CONV ---
        # We pad (ksize - 1) zeros to the past (left), and 0 to the future (right)
        self.pad = layers.ZeroPadding1D(padding=(ksize - 1, 0))
        self.dw_conv = layers.DepthwiseConv1D(
            kernel_size=ksize, 
            padding='valid', # Since we manually padded, we use 'valid'
            use_bias=False
        )
        
        self.bn1 = layers.BatchNormalization(momentum=0.95)
        self.eca = ECA()
        
        self.pw_conv = layers.Conv1D(filters=dim, kernel_size=1, use_bias=False)
        self.bn2 = layers.BatchNormalization(momentum=0.95)
        self.dropout = layers.Dropout(drop_rate)
        
    def call(self, inputs, training=False):
        # Forward Pass
        x = self.pad(inputs)
        x = self.dw_conv(x)
        
        x = self.bn1(x, training=training)
        x = tf.nn.swish(x)
        
        x = self.eca(x)
        
        x = self.pw_conv(x)
        x = self.bn2(x, training=training)
        x = self.dropout(x, training=training)
        
        return inputs + x

# --- EXECUTION ---
# Test the block locally to verify tensor flow and GPU stability
dummy_input = tf.random.normal([1, 64, 192]) # (Batch, Time, Dim)
conv_block = Conv1DBlock(dim=192, ksize=17)
output = conv_block(dummy_input)
print(f"Conv1DBlock output shape: {output.shape}")

Success: Memory Growth Enabled on 1 GPU(s).
Conv1DBlock output shape: (1, 64, 192)


# Step 3: The Transformer Block.

In [20]:
class TransformerBlock(layers.Layer):
    """
    Custom Transformer Block using Batch Normalization and Swish activation 
    to synergize with the Conv1D ECA blocks.
    """
    def __init__(self, dim, expand=2, num_heads=4, drop_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        # Self-Attention Mechanism
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim, dropout=drop_rate)
        self.bn1 = layers.BatchNormalization(momentum=0.95)
        self.drop1 = layers.Dropout(drop_rate)

        # Feed-Forward Network (FFN)
        self.ffn1 = layers.Dense(dim * expand, use_bias=False)
        self.ffn2 = layers.Dense(dim, use_bias=False)
        self.bn2 = layers.BatchNormalization(momentum=0.95)
        self.drop2 = layers.Dropout(drop_rate)

    def call(self, inputs, training=False):
        # 1. Attention Path with Residual Connection
        attn_out = self.mha(inputs, inputs, inputs, training=training)
        attn_out = self.drop1(attn_out, training=training)
        x1 = inputs + attn_out
        x1 = self.bn1(x1, training=training)

        # 2. Feed-Forward Path with Residual Connection
        ffn_out = self.ffn1(x1)
        ffn_out = tf.nn.swish(ffn_out)
        ffn_out = self.ffn2(ffn_out)
        ffn_out = self.drop2(ffn_out, training=training)
        
        x2 = x1 + ffn_out
        x2 = self.bn2(x2, training=training)

        return x2

# --- EXECUTION ---
transformer_block = TransformerBlock(dim=192)
# We pass the 'output' tensor generated from the Conv1DBlock in the previous step
transformer_output = transformer_block(output) 
print(f"TransformerBlock output shape: {transformer_output.shape}")

TransformerBlock output shape: (1, 64, 192)


# Step 4: The Model Assembly

In [21]:
class LateDropout(layers.Layer):
    """
    Custom layer that delays heavy dropout (p=0.8) until after a specific 
    training threshold to allow initial convergence.
    """
    def __init__(self, rate, start_step=0, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate
        self.start_step = start_step
        self.dropout = layers.Dropout(rate)
        # Counter to track training steps
        self.step_counter = tf.Variable(0, trainable=False, dtype=tf.int64)

    def call(self, inputs, training=False):
        if training:
            self.step_counter.assign_add(1)
            # Apply dropout only if the training step exceeds the start_step
            return tf.cond(
                self.step_counter > self.start_step,
                lambda: self.dropout(inputs, training=True),
                lambda: inputs
            )
        return inputs

def get_model(max_len=64, channels=1170, num_classes=250, dropout_step=0, dim=192):
    """
    End-to-End V2 Architecture matching Hoyeol Sohn's 1st Place Solution.
    """
    # 1170 channels = 130 landmarks * 9 spatial/temporal features
    inp = tf.keras.Input(shape=(max_len, channels), name='input_layer')
    
    # 1. Masking: Dynamically ignores padded zero-frames during training
    x = layers.Masking(mask_value=0.0, input_shape=(max_len, channels))(inp)
    
    ksize = 17
    
    # 2. Stem Block: Projects input features to the target hidden dimension (192)
    x = layers.Dense(dim, use_bias=False, name='stem_conv')(x)
    x = layers.BatchNormalization(momentum=0.95, name='stem_bn')(x)

    # 3. Stage 1: Local Context (CNN) + Global Context (Transformer)
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = TransformerBlock(dim, expand=2)(x)

    # 4. Stage 2: Local Context (CNN) + Global Context (Transformer)
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = Conv1DBlock(dim, ksize, drop_rate=0.2)(x)
    x = TransformerBlock(dim, expand=2)(x)

    # Note: We are bypassing the dim==384 block from the screenshot as we are 
    # building the dim=192 version to optimize batch sizes on an 8GB RTX 3070.


    # 5. Top Block: Feature expansion and pooling
    x = layers.Dense(dim * 2, activation=None, name='top_conv')(x)
    x = layers.GlobalAveragePooling1D()(x)
    
    # 6. Regularization & Classifier
    x = LateDropout(0.8, start_step=dropout_step)(x)
    
    # Outputting raw logits (no softmax) to use with CategoricalCrossentropy(from_logits=True)
    output = layers.Dense(num_classes, name='classifier')(x)
    
    return tf.keras.Model(inputs=inp, outputs=output)

# --- EXECUTION ---
v2_model = get_model(max_len=64, channels=1170, num_classes=250)
v2_model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_layer (InputLayer)    [(None, 64, 1170)]        0         
                                                                 
 masking_1 (Masking)         (None, 64, 1170)          0         
                                                                 
 stem_conv (Dense)           (None, 64, 192)           224640    
                                                                 
 stem_bn (BatchNormalization  (None, 64, 192)          768       
 )                                                               
                                                                 
 conv1d_block_9 (Conv1DBlock  (None, 64, 192)          41669     
 )                                                               
                                                                 
 conv1d_block_10 (Conv1DBloc  (None, 64, 192)          4166

# Step 5: The tf.data Pipeline and Sequence Padding

In [22]:
# 1. Load the Label Map
with open('../data/sign_to_prediction_index_map.json') as f:
    label_map = json.load(f)

train_df['label'] = train_df['sign'].map(label_map)

# 2. Group-Aware Train/Validation Split
gss = GroupShuffleSplit(test_size=0.1, n_splits=1, random_state=42)
train_idx, val_idx = next(gss.split(train_df, groups=train_df['participant_id']))

train_data = train_df.iloc[train_idx]
val_data = train_df.iloc[val_idx]

print(f"Training samples: {len(train_data)} | Validation samples: {len(val_data)}")

# 3. Data Generator Configuration
MAX_LEN = 64
CHANNELS = 1170
BATCH_SIZE = 64 

def load_video_and_pad(sequence_id):
    """Reads .npy, flattens spatial dims, truncates to 64, or pads with 0s to 64."""
    path = f"{SAVE_DIR}/{sequence_id.decode('utf-8')}.npy"
    frames = np.load(path)
    
    # --- THE FIX: Flatten (Time, 130, 9) to (Time, 1170) ---
    frames = frames.reshape(frames.shape[0], CHANNELS)
    
    length = frames.shape[0]
    if length < MAX_LEN:
        pad_size = MAX_LEN - length
        pad_array = np.zeros((pad_size, CHANNELS), dtype=np.float32)
        frames = np.concatenate([frames, pad_array], axis=0)
    elif length > MAX_LEN:
        frames = frames[:MAX_LEN]
        
    return frames.astype(np.float32)

def build_dataset(df, is_training=True):
    seq_ids = df['sequence_id'].astype(str).values
    labels = df['label'].values
    
    def map_fn(seq_id, label):
        frames = tf.numpy_function(func=load_video_and_pad, inp=[seq_id], Tout=tf.float32)
        frames.set_shape((MAX_LEN, CHANNELS)) 
        return frames, label

    dataset = tf.data.Dataset.from_tensor_slices((seq_ids, labels))
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=5000, reshuffle_each_iteration=True)
        
    dataset = dataset.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=is_training)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# 4. Instantiate the Datasets
train_dataset = build_dataset(train_data, is_training=True)
val_dataset = build_dataset(val_data, is_training=False)

# Verify the tensor shapes
for X_batch, y_batch in train_dataset.take(1):
    print(f"Batch X shape: {X_batch.shape} | Batch y shape: {y_batch.shape}")




    

Training samples: 80229 | Validation samples: 14248
Batch X shape: (64, 64, 1170) | Batch y shape: (64,)


# Step 6: Optimization, Callbacks, and Training

In [23]:
# 1. Global XLA Override: Instructs the C++ backend to skip JIT compilation entirely
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'
tf.config.optimizer.set_jit(False)

# 2. Custom Classes (Cooldown & LR Schedule)
class CooldownCallback(tf.keras.callbacks.Callback):
    """Pauses training at the end of each epoch to prevent thermal throttling on the RTX 3070."""
    def __init__(self, pause_seconds=45):
        super().__init__()
        self.pause_seconds = pause_seconds

    def on_epoch_end(self, epoch, logs=None):
        print(f"\n[Thermal Management] Pausing for {self.pause_seconds} seconds to cool down...")
        time.sleep(self.pause_seconds)

class WarmUpCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Bypasses local TF version errors by manually implementing warmup and cosine decay."""
    def __init__(self, initial_lr, target_lr, warmup_steps, total_steps, alpha=0.01):
        super().__init__()
        self.initial_lr = tf.cast(initial_lr, tf.float32)
        self.target_lr = tf.cast(target_lr, tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.decay_steps = tf.cast(total_steps - warmup_steps, tf.float32)
        self.alpha = tf.cast(alpha, tf.float32)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.initial_lr + (self.target_lr - self.initial_lr) * (step / self.warmup_steps)
        step_after_warmup = tf.maximum(step - self.warmup_steps, 0.0)
        decay_ratio = tf.minimum(step_after_warmup / self.decay_steps, 1.0)
        cosine_decay = 0.5 * (1.0 + tf.cos(tf.constant(math.pi) * decay_ratio))
        decayed_lr = (self.target_lr - self.target_lr * self.alpha) * cosine_decay + self.target_lr * self.alpha
        return tf.where(step < self.warmup_steps, warmup_lr, decayed_lr)

# 3. Calculate exact step intervals
EPOCHS = 50
BATCH_SIZE = 32 # Reduced for thermal management
steps_per_epoch = len(train_data) // BATCH_SIZE
total_steps = EPOCHS * steps_per_epoch
warmup_steps = int(total_steps * 0.1) 

lr_schedule = WarmUpCosineDecay(
    initial_lr=1e-5,
    target_lr=1e-4,  # <-- Reduced peak learning rate to prevent divergence
    warmup_steps=warmup_steps,
    total_steps=total_steps,
    alpha=1e-2
)

# 4. Optimizer Patch: Explicitly block the optimizer from calling XLA
try:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule, 
        weight_decay=1e-4,
        jit_compile=False
    )
except AttributeError:
    optimizer = tf.keras.optimizers.experimental.AdamW(
        learning_rate=lr_schedule, 
        weight_decay=1e-4,
        jit_compile=False
    )

# 5. Loss and Compilation
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

v2_model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['sparse_categorical_accuracy'],
    jit_compile=False 
)

# 6. Callbacks List
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath='../models/asl_v2_weights_best.h5',
        monitor='val_sparse_categorical_accuracy',
        save_best_only=True,
        save_weights_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_sparse_categorical_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir='./logs_v2', 
        histogram_freq=1
    ),
    CooldownCallback(pause_seconds=45) # Enforces the thermal cooldown
]

# 7. Launch Local GPU Training
print("Launching V2 Training on local RTX 3070...")
history = v2_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

Launching V2 Training on local RTX 3070...
Layer Conv1DBlock has arguments ['dim', 'ksize', 'drop_rate']
in `__init__` and therefore must override `get_config()`.

Example:

class CustomLayer(keras.layers.Layer):
    def __init__(self, arg1, arg2):
        super().__init__()
        self.arg1 = arg1
        self.arg2 = arg2

    def get_config(self):
        config = super().get_config()
        config.update({
            "arg1": self.arg1,
            "arg2": self.arg2,
        })
        return config
Epoch 1/50
1253/1253 [==============================] - ETA: 0s - loss: 5.5883 - sparse_categorical_accuracy: 0.0048
Epoch 1: val_sparse_categorical_accuracy improved from -inf to 0.00379, saving model to asl_v2_weights_best.h5

[Thermal Management] Pausing for 45 seconds to cool down...
1253/1253 [==============================] - 123s 93ms/step - loss: 5.5883 - sparse_categorical_accuracy: 0.0048 - val_loss: 5.5187 - val_sparse_categorical_accuracy: 0.0038
Epoch 2/50
1252/1253 [=====

In [25]:
# 1. Recreate the V2 architecture skeleton
v2_model = get_model(max_len=64, channels=1170, num_classes=250)

# 2. Load the best weights captured by the ModelCheckpoint callback
v2_model.load_weights('../models/asl_v2_weights_best.h5')
print("Successfully loaded optimal weights from asl_v2_weights_best.h5!")

# 3. Evaluate Top-1 and Top-5 Accuracy on the Validation Set
top1_acc = tf.keras.metrics.SparseCategoricalAccuracy()
top5_acc = tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)

print("Running validation evaluation...")
for x_batch, y_batch in val_dataset:
    preds = v2_model.predict(x_batch, verbose=0)
    top1_acc.update_state(y_batch, preds)
    top5_acc.update_state(y_batch, preds)

print(f"\n--- V2 Model Results ---")
print(f"Top-1 Validation Accuracy: {top1_acc.result().numpy() * 100:.2f}%")
print(f"Top-5 Validation Accuracy: {top5_acc.result().numpy() * 100:.2f}%")

Successfully loaded optimal weights from asl_v2_weights_best.h5!
Running validation evaluation...

--- V2 Model Results ---
Top-1 Validation Accuracy: 9.95%
Top-5 Validation Accuracy: 24.99%
